In [1]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Load Environment Variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API Client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [4]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text


In [5]:
# Strutured output using output_config - replaces the prefill pattern

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Extract structured data from this text: Acme Corp is a manufacturing firm based in Detroit with 200 employees, founded in 1987."
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "client": {"type": "string"},
                    "industry": {"type": "string"},
                    "location": {"type": "string"},
                    "employee_count": {"type": "integer"},
                    "year_founded": {"type": "integer"}
                },
                "required": ["client", "industry", "location", "employee_count", "year_founded"],
                "additionalProperties": False
            }
        }
    }       
)
print(response.content[0].text)

{"client":"Acme Corp","industry":"manufacturing","location":"Detroit","employee_count":200,"year_founded":1987}


In [6]:
# Import and validate the structured output

import json

data = json.loads(response.content[0].text)
print(f"Company: {data['client']}")
print(f"Industry: {data['industry']}")
print(f"Location: {data['location']}")
print(f"Employees: {data['employee_count']}")
print(f"Founded: {data['year_founded']}")

Company: Acme Corp
Industry: manufacturing
Location: Detroit
Employees: 200
Founded: 1987


In [7]:
# Extract structured findings from unstructured consulting notes

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": """Meeting with Riverside Health System operations team. Key issues identified:
                        - Patient discharge process averaging 4 hours, benchmark is 90 minutes
                        - Three separate EHR systems not integrated, staff manually reconciling records
                        - Nursing staff turnover at 34% annually, industry average is 22%
                        - Finance team estimates manual reconciliation costs 2.3M annually
                        - CEO wants a 90-day improvement roadmap
                        Priority appears to be EHR integration based on downstream impact on both discharge time and staff burden.
             """
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "client": {"type": "string"},
                    "top_priority": {"type": "string"},
                    "actions": {"type": "array", "items": {"type": "string"}},
                    "problems": {"type": "array", "items": {"type": "string"}},
                    "kpis": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "current": {"type": "string"},
                                "target": {"type": "string"}
                            },
                            "required": ["current", "target"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["client", "top_priority", "actions", "problems", "kpis"],
                "additionalProperties": False
            }
        }
    }       
)
print(response.content[0].text)

{"client":"Riverside Health System","top_priority":"EHR system integration to reduce downstream impact on discharge times and staff burden","actions":["Develop a 90-day improvement roadmap aligned with CEO directive","Conduct technical assessment of all three EHR systems to identify integration architecture options","Engage EHR vendors to evaluate APIs and interoperability capabilities","Design and implement automated data reconciliation workflows to eliminate manual processes","Launch nursing staff retention initiative targeting reduction of turnover from 34% to at or below 22%","Establish a discharge process redesign task force with target of reducing average discharge time from 4 hours to 90 minutes"],"problems":["Patient discharge process averaging 4 hours against a 90-minute industry benchmark","Three non-integrated EHR systems requiring costly and time-consuming manual reconciliation","Nursing staff turnover at 34% annually versus 22% industry average indicating retention crisis"

In [8]:
# Import and validate the structured output

import json

data = json.loads(response.content[0].text)
print(f"Company: {data['client']}")
print(f"Priority: {data['top_priority']}")
print("\nActions:")
for action in data['actions']:
    print(f"- {action}")
print("\nProblems:")
for problem in data['problems']:
    print(f"- {problem}")
print("\nKPIs:")
for kpi in data['kpis']:
    print(f"- Current: {kpi['current']}, Target: {kpi['target']}")


Company: Riverside Health System
Priority: EHR system integration to reduce downstream impact on discharge times and staff burden

Actions:
- Develop a 90-day improvement roadmap aligned with CEO directive
- Conduct technical assessment of all three EHR systems to identify integration architecture options
- Engage EHR vendors to evaluate APIs and interoperability capabilities
- Design and implement automated data reconciliation workflows to eliminate manual processes
- Launch nursing staff retention initiative targeting reduction of turnover from 34% to at or below 22%
- Establish a discharge process redesign task force with target of reducing average discharge time from 4 hours to 90 minutes

Problems:
- Patient discharge process averaging 4 hours against a 90-minute industry benchmark
- Three non-integrated EHR systems requiring costly and time-consuming manual reconciliation
- Nursing staff turnover at 34% annually versus 22% industry average indicating retention crisis
- Manual rec